# Basic RAG chain

Query -> retrieve (pgvector, direct SQL) -> generate (LangChain LCEL + GPT-4o-mini).

Retrieves from the `chunks` table (OpenAI `text-embedding-3-small` embeddings) ingested by `ingest_pgvector.py` / `embed_missing.py`. Swap the table name to `chunks_nomic` to compare against local `nomic-embed-text` embeddings.

**Prerequisite:** `docker compose up -d` (Postgres+pgvector container `rag-postgres`).

In [1]:
import os

import psycopg
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

DSN = "host=localhost port=5432 dbname=rag user=rag password=rag"
TABLE = "chunks"  # or "chunks_nomic" to compare embedding models
EMB_MODEL = "text-embedding-3-small"

embeddings_model = OpenAIEmbeddings(model=EMB_MODEL, api_key=os.environ["OPENAI_KEY"])

## Retrieval

`chunks`/`chunks_nomic` have a custom flat schema (not LangChain's managed `PGVector` collection), so retrieval is a direct SQL query using pgvector's `<=>` cosine-distance operator, filtered optionally by `stock_symbol` and/or `section`.

In [2]:
def retrieve(question: str, k: int = 5, stock_symbol: str = None, section: str = None) -> list[dict]:
    """Embed the question and return the top-k most similar chunks from Postgres."""
    query_embedding = embeddings_model.embed_query(question)
    embedding_str = "[" + ",".join(str(x) for x in query_embedding) + "]"

    where_clauses = []
    params = {"embedding": embedding_str, "k": k}
    if stock_symbol:
        where_clauses.append("stock_symbol = %(stock_symbol)s")
        params["stock_symbol"] = stock_symbol
    if section:
        where_clauses.append("section = %(section)s")
        params["section"] = section
    where_sql = f"WHERE {' AND '.join(where_clauses)}" if where_clauses else ""

    sql = f"""
        SELECT chunk_id, doc_id, stock_symbol, section, text,
               embedding <=> %(embedding)s::vector AS distance
        FROM {TABLE}
        {where_sql}
        ORDER BY distance
        LIMIT %(k)s
    """

    with psycopg.connect(DSN) as conn, conn.cursor(row_factory=psycopg.rows.dict_row) as cur:
        cur.execute(sql, params)
        return cur.fetchall()


def format_chunks(chunks: list[dict]) -> str:
    return "\n\n---\n\n".join(
        f"[{c['section']} | {c['doc_id']}]\n{c['text']}" for c in chunks
    )

## Generation chain (LCEL)

`retrieve` + `format_chunks` plug into a LangChain `Runnable` pipeline alongside the prompt and LLM.

In [3]:
PROMPT_TEMPLATE = """\
You are a financial analyst assistant. Answer the question using only the provided excerpts \
from the company's 10-K annual filing. If the answer is not in the excerpts, say so.

Excerpts:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ["OPENAI_KEY"])


def make_rag_chain(k: int = 5, stock_symbol: str = None, section: str = None):
    """Build an LCEL RAG chain: retrieve -> format -> prompt -> llm -> parse."""

    def retrieve_and_format(question: str) -> str:
        chunks = retrieve(question, k=k, stock_symbol=stock_symbol, section=section)
        return format_chunks(chunks)

    return (
        {"context": RunnableLambda(retrieve_and_format), "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

## Ask questions

Ticker is one of: aapl, amd, avgo, ba, brka, cost, crsp, dpz, goog, ibm, intc, isrg, meta, mu, nvda, smr, snps, tsla, unh.

In [4]:
chain = make_rag_chain(k=5, stock_symbol="aapl")

answer = chain.invoke("What are Apple's main products and services?")
print(answer)

Apple's main products include smartphones (iPhone), personal computers (Mac), multipurpose tablets (iPad), wearables (smartwatches, wireless headphones, and spatial computers), and accessories. 

Apple's main services include advertising services, AppleCare (service and support products), cloud services, digital content (such as the App Store and subscription-based services like Apple Arcade, Apple Fitness+, Apple Music, Apple News+, and Apple TV), and payment services (Apple Card and Apple Pay).


In [5]:
# Inspect retrieved chunks to understand what the LLM saw
chunks = retrieve("What are the main risk factors?", k=5, stock_symbol="aapl")

for i, c in enumerate(chunks, 1):
    print(f"--- Chunk {i} | {c['section']} | {c['doc_id']} | distance={c['distance']:.4f} ---")
    print(c["text"][:400])
    print()

--- Chunk 1 | Item 1A. | aapl-20250927 | distance=0.6242 ---
Item 1A. Risk Factors
The following summarizes factors that could have a material adverse effect on the Companys business, reputation, results of operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this section are based on the Companys beliefs and opinions regarding matters that could materially adverse

--- Chunk 2 | Item 7A. | aapl-20250927 | distance=0.6974 ---
Item 7A. Quantitative and Qualitative Disclosures About Market Risk
The Company is exposed to economic risk from interest rates and foreign exchange rates. The Company uses various strategies to manage these risks; however, they may still impact the Companys consolidated financial statements.
Interest Rate Risk
The Company is primarily exposed to fluctuations in U.S. interest rates and their impac

--- Chunk 3 | Item 1A. | aapl-20250927 | distance=0.7087 ---
The Companys 